##Evidencia 1
Carlos Alonso Gómez Reyes - A00841169
José Manuel Pulido Pérez - A01178806
Jaime Emilio Garza Rodríguez- a00843320

# Entrega 1: Descripción y Análisis Inicial de los Datos

Este notebook está organizado en las 5 instrucciones de la Entrega 1, cada una en su propia sección claramente marcada:

1. Adquisición y Limpieza de Datos (2 pts)
2. Identificación de Variables Dependientes e Independientes (2 pts)
3. Evaluación de Tecnologías de Información (2 pts)
4. Análisis del Perfil de Riesgo del Cliente (2 pts)
5. Primera Visualización y Descripción de los Datos (2 pts)

## Import Libraries

Primero importamos las librerías necesarias para manipulación de datos, manejo de archivos e interacción con la base de datos.

In [ ]:
import pandas as pd
import numpy as np
import os
import sqlite3
from google.colab import files

# 1. Adquisición y Limpieza de Datos (2 pts)

**Adquisición:** se utilizan los históricos de precio diario de 10 acciones (AMD, Apple, Coca-Cola, Intel, Meta, Micron, Microsoft, NVIDIA, Taiwan Semiconductor y Walmart), exportados como CSV a partir de un proveedor de datos financieros (formato equivalente al de Yahoo Finance / Investing.com: Fecha, Apertura, Máximo, Mínimo, Cierre, Volumen y Cambio %) y cargados a Colab.

**Limpieza:** se convierten los tipos de dato, se eliminan duplicados, se interpolan/eliminan valores faltantes y se descartan precios o volúmenes inválidos, dejando un DataFrame listo para el análisis.

### 1.1 Carga y combinación de archivos CSV

Importamos los CSV's de precio de las 10 acciones seleccionadas.

In [ ]:
# List of files in the current directory

csv_files = [
    '/content/AMD Stock Price History.csv',
    '/content/Apple Stock Price History.csv',
    '/content/Coca-Cola Stock Price History.csv',
    '/content/Intel Stock Price History.csv',
    '/content/Meta Platforms Stock Price History.csv',
    '/content/Micron Stock Price History.csv',
    '/content/Microsoft Stock Price History.csv',
    '/content/NVIDIA Stock Price History.csv',
    '/content/Taiwan Semiconductor Stock Price History.csv',
    '/content/Walmart Stock Price History.csv'
]

all_dfs = []

for filepath in csv_files:
    filename = os.path.basename(filepath)
    print(f"Processing file: {filename}")
    try:
        # Determine the Ticker from the filename (e.g., 'Apple Stock Price History.csv' -> 'Apple')
        ticker = filename.split(' Stock')[0]

        # Read the CSV content into a pandas DataFrame directly from the path
        df_ticker = pd.read_csv(filepath)

        # Add the 'Ticker' column
        df_ticker['Ticker'] = ticker

        all_dfs.append(df_ticker)
    except Exception as e:
        print(f"Error processing {filename}: {e}")

# Concatenate all DataFrames into a single one
if all_dfs:
    df = pd.concat(all_dfs, ignore_index=True)
    print("\nAll files loaded and combined successfully!")
    print("Initial DataFrame head:")
    display(df.head())
    print(f"Initial DataFrame shape: {df.shape}")
else:
    print("No files were loaded or processed.")
    df = pd.DataFrame()  # Create an empty DataFrame if no files were loaded

### 1.2 Limpieza de los datos

Se realizan las siguientes operaciones de limpieza sobre el DataFrame combinado:
- Conversión de tipos de datos (fechas, volumen con sufijos `M`/`K`, cambio % con símbolo `%`).
- Eliminación de filas duplicadas (mismo Ticker y Fecha).
- Manejo de valores faltantes: interpolación temporal acotada y eliminación de filas que no se pudieron completar.
- Eliminación de precios inválidos (negativos o cero) y volúmenes negativos.

In [ ]:
if not df.empty:
    # --- 1.2.1 Conversión de tipos de datos ---
    # Convert 'Date' to datetime, coercing errors to NaT (Not a Time)
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

    # Drop rows where 'Date' could not be parsed
    initial_rows_date_na = len(df)
    df.dropna(subset=['Date'], inplace=True)
    date_na_count = initial_rows_date_na - len(df)
    if date_na_count > 0:
        print(f"Dropped {date_na_count} rows due to invalid 'Date' format.")

    # Preprocessing 'Vol.' column (e.g., '17.07M' -> 17070000)
    def clean_volume(vol_str):
        if pd.isna(vol_str) or vol_str == '':  # Handle empty string explicitly
            return np.nan
        vol_str = str(vol_str).strip()
        if 'M' in vol_str:
            try:
                return float(vol_str.replace('M', '')) * 1_000_000
            except ValueError:
                return np.nan
        elif 'K' in vol_str:
            try:
                return float(vol_str.replace('K', '')) * 1_000
            except ValueError:
                return np.nan
        try:  # Try direct conversion for numbers without M/K
            return float(vol_str)
        except ValueError:  # Catch other non-numeric strings
            return np.nan

    # Preprocessing 'Change %' column (e.g., '4.91%' -> 4.91)
    def clean_percentage(pct_str):
        if pd.isna(pct_str) or pct_str == '':  # Handle empty string explicitly
            return np.nan
        pct_str = str(pct_str).strip().replace('%', '')
        try:  # Try direct conversion
            return float(pct_str)
        except ValueError:  # Catch other non-numeric strings
            return np.nan

    # Apply cleaning functions to specific columns
    if 'Vol.' in df.columns:
        df['Vol.'] = df['Vol.'].apply(clean_volume)
        print("Cleaned 'Vol.' column by converting 'M' and 'K' to numeric and handling other formats.")
    if 'Change %' in df.columns:
        df['Change %'] = df['Change %'].apply(clean_percentage)
        print("Cleaned 'Change %' column by removing '%' and converting to numeric.")

    # Convert remaining numeric columns, coercing non-numeric values to NaN
    numeric_cols = ['Open', 'High', 'Low', 'Price', 'Vol.', 'Change %']
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    print("Converted numeric columns to appropriate types.")

    # --- 1.2.2 Eliminación de filas duplicadas ---
    initial_rows_dup = len(df)
    df.sort_values(by=['Ticker', 'Date'], inplace=True)  # Sort to ensure consistent duplicate handling
    df.drop_duplicates(subset=['Ticker', 'Date'], inplace=True)
    duplicates_count = initial_rows_dup - len(df)
    if duplicates_count > 0:
        print(f"Removed {duplicates_count} duplicate rows (same Ticker and Date).")

    # --- 1.2.3 Manejo de valores faltantes (Interpolación y Eliminación) ---
    initial_nan_count_total = df.isnull().sum().sum()  # Total NaNs before interpolation

    # Columns for interpolation (numeric columns that can have NaNs)
    cols_for_interpolation = ['Open', 'High', 'Low', 'Price', 'Vol.']

    # Set 'Date' as index temporarily for time-based interpolation
    df_temp = df.set_index('Date')

    # Apply interpolation *only* to the specified numeric columns within each Ticker group
    for col_interp in cols_for_interpolation:
        if col_interp in df_temp.columns:
            df_temp[col_interp] = df_temp.groupby('Ticker')[col_interp].transform(
                lambda g: g.interpolate(method='time', limit_direction='both', limit=3, limit_area='inside')
            )

    df_cleaned = df_temp.reset_index()

    nan_after_interpolation_total = df_cleaned.isnull().sum().sum()
    interpolated_nan_count = initial_nan_count_total - nan_after_interpolation_total
    print(f"Interpolated NaNs: {interpolated_nan_count} values filled.")

    # Drop remaining rows with NaNs in *critical* columns that couldn't be filled
    critical_cols_for_dropna = ['Date', 'Open', 'High', 'Low', 'Price', 'Vol.']
    nan_in_critical_cols_before_drop = df_cleaned[critical_cols_for_dropna].isnull().sum().sum()

    if nan_in_critical_cols_before_drop > 0:
        initial_rows_before_critical_drop = len(df_cleaned)
        df_cleaned.dropna(subset=critical_cols_for_dropna, inplace=True)
        rows_dropped_critical = initial_rows_before_critical_drop - len(df_cleaned)
        if rows_dropped_critical > 0:
            print(f"Dropped {rows_dropped_critical} rows that still contained NaN values in critical columns after interpolation.")
    else:
        print("No NaN values remaining in critical price/volume/date columns after interpolation.")

    df = df_cleaned  # Update the main DataFrame after cleaning NaNs

    # --- 1.2.4 Eliminación de precios inválidos (negativos o cero) y volúmenes negativos ---
    initial_rows_before_invalid_filter = len(df)

    price_cols = ['Open', 'High', 'Low', 'Price']
    for col in price_cols:
        if col in df.columns:
            df = df[df[col] > 0]

    if 'Vol.' in df.columns:
        df = df[df['Vol.'] >= 0]

    invalid_price_volume_count = initial_rows_before_invalid_filter - len(df)
    if invalid_price_volume_count > 0:
        print(f"Removed {invalid_price_volume_count} rows with invalid (negative or zero) prices or negative volumes.")

    print("\n--- Limpieza de Datos Completada ---")

    # Sort the final DataFrame by Ticker and Date
    df.sort_values(by=['Ticker', 'Date'], inplace=True)
    df.reset_index(drop=True, inplace=True)
    print("DataFrame sorted by Ticker and Date.")

else:
    print("No DataFrame to clean as no files were loaded.")

### 1.3 Resultado esperado: DataFrame limpio almacenado en la base de datos "fondo"

Guardamos el DataFrame ya limpio (precios OHLC, volumen y % de cambio por Ticker y Fecha) en la base de datos SQLite `fondo.db`, tabla `precios`. Esta tabla es la que alimentará el resto del análisis.

In [ ]:
if not df.empty:
    print("DataFrame limpio (head):")
    display(df.head())
    print(f"Forma final del DataFrame limpio: {df.shape}")

    # --- Almacenamiento en SQLite ---
    db_name = 'fondo.db'
    table_name = 'precios'

    conn = sqlite3.connect(db_name)
    try:
        cursor = conn.cursor()
        cursor.execute(f"DROP TABLE IF EXISTS {table_name}")
        conn.commit()

        df.to_sql(table_name, conn, if_exists='replace', index=False)
        print(f"\nDataFrame guardado exitosamente en SQLite '{db_name}' en la tabla '{table_name}'.")
    except Exception as e:
        print(f"Error al guardar en SQLite: {e}")
    finally:
        conn.close()

    # --- Exportar a CSV de respaldo ---
    csv_filename = 'cleaned_prices.csv'
    try:
        df.to_csv(csv_filename, index=False)
        print(f"DataFrame de respaldo exportado a '{csv_filename}'.")
        # files.download(csv_filename)  # Descomentar para descargar automáticamente
    except Exception as e:
        print(f"Error al exportar a CSV: {e}")
else:
    print("No DataFrame to save or export as no files were loaded.")

# 2. Identificación de Variables Dependientes e Independientes (2 pts)

Se parte de la tabla `precios` (ya limpia) y se calcula la variable dependiente (retorno diario) y 4 variables independientes, cada una justificada con una teoría financiera:

| Variable | Tipo | Justificación teórica |
|---|---|---|
| `Daily_Return` | Y (dependiente) | Retorno diario porcentual: es la variable que se busca explicar/predecir. |
| `Rendimiento_Mercado` | X1 | CAPM (Sharpe, 1964): el retorno promedio del mercado (proxy: promedio de las 10 acciones) captura el riesgo sistemático que afecta a todos los activos. |
| `Volatilidad_20d` | X2 | Teoría de portafolios de Markowitz: la desviación estándar móvil de 20 días es la medida estándar de riesgo/varianza de un activo. |
| `Volumen_Log` | X3 | Teoría de liquidez de mercado: el volumen (transformado con log para reducir asimetría) es un proxy de liquidez y de la información que absorbe el precio. |
| `Momentum_10d` | X4 | Anomalías de mercado / efecto momentum (Jegadeesh & Titman, 1993): el retorno promedio de los últimos 10 días captura la persistencia de tendencias de corto plazo. |

In [ ]:
# Punto 2: Variables dependientes e independientes

conn = sqlite3.connect("fondo.db")
df = pd.read_sql("SELECT * FROM precios", conn, parse_dates=["Date"])
conn.close()
df = df.sort_values(["Ticker", "Date"]).reset_index(drop=True)

# Y: Daily_Return - retorno diario porcentual (variable dependiente), calculado sobre Price
df["Daily_Return"] = df.groupby("Ticker")["Price"].pct_change() * 100

# Marcamos retornos diarios atipicos (> 40% en valor absoluto) para la descripcion posterior
outlier_threshold = 40
df["Is_Return_Outlier"] = df["Daily_Return"].abs() > outlier_threshold

# X1: Rendimiento_Mercado - promedio diario de las 10 acciones (CAPM)
mercado = df.groupby("Date")["Daily_Return"].mean().rename("Rendimiento_Mercado")
df = df.merge(mercado, on="Date", how="left")

# X2: Volatilidad_20d - riesgo (desv. estandar movil de 20 dias)
df["Volatilidad_20d"] = df.groupby("Ticker")["Daily_Return"].transform(
    lambda s: s.rolling(20, min_periods=10).std()
)

# X3: Volumen_Log - liquidez (log del volumen)
df["Volumen_Log"] = np.log1p(df["Vol."])

# X4: Momentum_10d - anomalias de mercado (retorno promedio de 10 dias)
df["Momentum_10d"] = df.groupby("Ticker")["Daily_Return"].transform(
    lambda s: s.rolling(10, min_periods=5).mean()
)

print(df[["Ticker", "Date", "Daily_Return", "Rendimiento_Mercado",
          "Volatilidad_20d", "Volumen_Log", "Momentum_10d"]].tail())

resumen = pd.DataFrame([
    ("Daily_Return", "Y", "Retorno diario a explicar/predecir"),
    ("Rendimiento_Mercado", "X1", "CAPM: riesgo sistematico de mercado"),
    ("Volatilidad_20d", "X2", "Varianza como medida de riesgo"),
    ("Volumen_Log", "X3", "Liquidez de mercado"),
    ("Momentum_10d", "X4", "Anomalia de mercado / momentum"),
], columns=["Variable", "Tipo", "Justificacion"])
print(resumen.to_string(index=False))

conn = sqlite3.connect("fondo.db")
df.to_sql("variables_modelo", conn, if_exists="replace", index=False)
conn.close()
df.to_csv("variables_modelo.csv", index=False)
print("Guardado en fondo.db (tabla variables_modelo) y variables_modelo.csv")

# 3. Evaluación de Tecnologías de Información (2 pts)

Se compara Python, R y Excel con una matriz de criterios ponderados (automatización, integración con APIs, Machine Learning, manejo de grandes datos y facilidad de uso), y se calcula el puntaje ganador.

In [ ]:
# Punto 3: Evaluacion de tecnologias (Python vs R vs Excel)

criterios = pd.DataFrame({
    "Criterio": ["Automatizacion", "Integracion con Interactive Brokers/APIs",
                 "Machine Learning", "Manejo de grandes datos", "Facilidad de uso"],
    "Peso": [0.25, 0.25, 0.25, 0.15, 0.10],
    "Python": [5, 5, 5, 5, 3],
    "R":      [4, 3, 3, 4, 3],
    "Excel":  [1, 1, 1, 2, 5],
})

for col in ["Python", "R", "Excel"]:
    criterios[f"{col}_pond"] = criterios[col] * criterios["Peso"]

print(criterios)

totales = criterios[["Python_pond", "R_pond", "Excel_pond"]].sum()
print("\nPuntaje final (0-5):")
print(totales)

ganador = totales.idxmax().replace("_pond", "")
print(f"\nTecnologia seleccionada: {ganador}")

**Justificación:** Python gana por su ecosistema de Machine Learning, mayor facilidad de automatización y mejor manejo de grandes volúmenes de datos, mientras que Excel no escala ni se automatiza bien, y R, aunque es fuerte en estadística, queda por debajo en automatización y ML.

# 4. Análisis del Perfil de Riesgo del Cliente (2 pts)

Se define un cliente hipotético (cuestionario de 5 preguntas), se calcula su perfil (Conservador/Moderado/Agresivo) y se sugiere qué acciones del portafolio le convienen según su volatilidad.

In [ ]:
# Punto 4: Perfil de riesgo del cliente
# Usa pd y sqlite3 ya importados al inicio del notebook.

# Cliente hipotetico (cuestionario simple, 1-5 c/u)
cliente = {
    "Horizonte de inversion (largo=5)": 4,
    "Tolerancia a perdidas (alta=5)": 3,
    "Estabilidad de ingresos (alta=5)": 4,
    "Conocimiento financiero (alto=5)": 3,
    "Objetivo (crecimiento=5, preservar=1)": 4,
}

puntaje = sum(cliente.values()) / len(cliente)

if puntaje < 2.5:
    perfil = "Conservador"
elif puntaje < 3.8:
    perfil = "Moderado"
else:
    perfil = "Agresivo"

print(f"Puntaje promedio: {puntaje:.2f} -> Perfil: {perfil}")

# Adaptar el portafolio segun el perfil, usando la volatilidad ya calculada
conn = sqlite3.connect("fondo.db")
vol = pd.read_sql("SELECT Ticker, Volatilidad_20d FROM variables_modelo", conn)
conn.close()

vol_prom = vol.groupby("Ticker")["Volatilidad_20d"].mean().sort_values()

if perfil == "Conservador":
    seleccion = vol_prom.head(5)
elif perfil == "Agresivo":
    seleccion = vol_prom.tail(5)
else:
    seleccion = vol_prom  # las 10, sin filtrar

print(f"\nAcciones sugeridas para perfil {perfil} (por volatilidad):")
print(seleccion)

El cliente hipotético obtuvo un puntaje de 3.60/5, lo que lo ubica en un perfil Moderado.

Se muestran las 10 acciones ordenadas por volatilidad, para que el asesor pueda diversificar y ponderar en vez de excluir.

Para este cliente Moderado, la recomendación natural es un portafolio balanceado, sobreponderando las acciones defensivas de baja volatilidad y usando las de semiconductores en menor proporción como complemento de crecimiento, en vez de concentrarse en un solo extremo del espectro de riesgo.

# 5. Primera Visualización y Descripción de los Datos (2 pts)

Evolución de precios (base 100, por acción), distribución de retornos diarios (por acción), retorno acumulado del portafolio, y riesgo vs. retorno anualizado.

In [ ]:
# Punto 5: Visualizacion inicial (acciones y portafolio)
import matplotlib.pyplot as plt

conn = sqlite3.connect("fondo.db")
df = pd.read_sql("SELECT * FROM variables_modelo", conn, parse_dates=["Date"])
conn.close()

tickers = sorted(df["Ticker"].unique())
colores = plt.get_cmap("tab10").colors

# 1. Evolucion de precios normalizada (base 100) - por accion
fig, ax = plt.subplots(figsize=(10, 5))
for i, t in enumerate(tickers):
    serie = df[df["Ticker"] == t].sort_values("Date")
    precio_norm = serie["Price"] / serie["Price"].iloc[0] * 100
    ax.plot(serie["Date"], precio_norm, label=t, color=colores[i % 10], linewidth=1.5)
ax.set_title("Evolucion de precios (base 100)")
ax.legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.savefig("01_precios.png", dpi=150)
plt.show()

# 2. Distribucion de retornos diarios - por accion (small multiples)
fig, axes = plt.subplots(2, 5, figsize=(15, 5), sharex=True)
for ax, t in zip(axes.flat, tickers):
    serie = df[df["Ticker"] == t]["Daily_Return"].dropna()
    ax.hist(serie, bins=30, color="steelblue")
    ax.set_title(t, fontsize=9)
fig.suptitle("Distribucion de retornos diarios por accion")
plt.tight_layout()
plt.savefig("02_distribucion_retornos.png", dpi=150)
plt.show()

# 3. Retorno acumulado del portafolio (equal-weighted, via Rendimiento_Mercado)
portafolio = df.drop_duplicates("Date").sort_values("Date")[["Date", "Rendimiento_Mercado"]]
portafolio["Retorno_Acumulado"] = (1 + portafolio["Rendimiento_Mercado"] / 100).cumprod()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(portafolio["Date"], portafolio["Retorno_Acumulado"], color=colores[0], linewidth=2)
ax.set_title("Retorno acumulado del portafolio (10 acciones, ponderacion igual)")
plt.tight_layout()
plt.savefig("03_portafolio_acumulado.png", dpi=150)
plt.show()

# 4. Riesgo vs retorno por accion (anualizado)
resumen = df.groupby("Ticker")["Daily_Return"].agg(["mean", "std"]).reset_index()
resumen["Retorno_Anual"] = resumen["mean"] * 252
resumen["Volatilidad_Anual"] = resumen["std"] * np.sqrt(252)

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(resumen["Volatilidad_Anual"], resumen["Retorno_Anual"], color=colores[3], s=60)
for _, row in resumen.iterrows():
    ax.annotate(row["Ticker"], (row["Volatilidad_Anual"], row["Retorno_Anual"]), fontsize=8)
ax.set_xlabel("Volatilidad anualizada (%)")
ax.set_ylabel("Retorno anualizado (%)")
ax.set_title("Riesgo vs. retorno por accion")
plt.tight_layout()
plt.savefig("04_riesgo_retorno.png", dpi=150)
plt.show()

print("Graficos guardados: 01_precios.png, 02_distribucion_retornos.png,",
      "03_portafolio_acumulado.png, 04_riesgo_retorno.png")

Precios: el portafolio está dominado por Micron e Intel, que se disparan a partir de finales de 2025, mientras Apple, Coca-Cola, Meta, Microsoft y Walmart se mantienen casi planos.

Distribución de retornos: todas centradas en 0 con forma de campana, pero Micron, Intel y AMD tienen colas más anchas consistente con ser las más riesgosas mientras Apple, Coca-Cola y Microsoft son mucho más angostas.

Portafolio acumulado: crece de 1.0 a ~2.6, con el salto fuerte justo cuando despegan Micron/Intel el rendimiento del portafolio está siendo "jalado" por esas dos acciones, no por diversificación real.

Riesgo-retorno: confirma todo lo anterior Micron e Intel están arriba a la derecha, el resto de defensivas (Coca-Cola, Microsoft, Walmart) abajo a la izquierda. No hay ninguna acción "ideal" (bajo riesgo + alto retorno).